# 원판 가림의 채움 방식 비교

기존 회색 가림 모델은 회색으로 가린 검증 프레임에서 기존 모델보다 7장을 더 찾았지만, 같은 위치를 바닥 질감으로 채우자 차이가 2장으로 줄었습니다. 이번에는 **학습 가림의 채움 방식만** 바꾸고, 보지 못한 패턴에서도 효과가 남는지 확인합니다.

두 가림 실험의 적용 확률 0.30, 박스 안쪽 면적 25~40%, 위치 추출 방법과 위치 시드를 고정합니다. 질감은 학습 프레임에서 정답 박스와 겹치지 않는 영역을 잘라 모읍니다. 질감 선택 난수를 따로 사용해 가림 위치 추출에 영향을 주지 않습니다.

In [ ]:
from functools import partial
from random import Random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from PIL import Image
from matplotlib.patches import Rectangle
from ultralytics import YOLO

from wlcv.config import ANNOTATIONS, DATASET, RUNS, SPLIT, VIDEOS
from wlcv.data.dataset import prepare_dataset
from wlcv.data.plate_labels import read_boxes
from wlcv.evaluation.occlusion_study import validation_occlusion_scores
from wlcv.training.background_textures import resize_background, sample_training_backgrounds
from wlcv.training.plate_occlusion import apply_plate_occlusion, train_occlusion

baseline_dir = RUNS / "plate_baseline"
gray_dir = RUNS / "plate_occlusion"
candidate_dir = RUNS / "plate_occlusion_texture"

## 1. 같은 위치의 회색과 배경 질감 비교

학습 프레임 한 장에서 같은 위치·크기로 가리고, 채움만 바뀌는지 봅니다. 예시에서는 변화를 항상 볼 수 있게 확률을 1로 설정합니다. 실제 학습 확률은 0.30입니다.

In [ ]:
stem = "a01_sn_f_01_f13"
with Image.open(DATASET / "images" / "train" / f"{stem}.jpg") as image:
    frame = np.array(image.convert("RGB"))

boxes = read_boxes(DATASET / "labels" / "train" / f"{stem}.txt")
images = torch.from_numpy(frame.copy()).permute(2, 0, 1)[None].float() / 255
patches = sample_training_backgrounds(DATASET)
print("학습에서 가져온 질감 수", len(patches))
texture_fill = partial(resize_background, patches, Random(1))
gray_images = images.clone()
apply_plate_occlusion(gray_images, [0] * len(boxes), boxes, Random(0), probability=1)
apply_plate_occlusion(images, [0] * len(boxes), boxes, Random(0), probability=1, make_fill=texture_fill)

height, width = frame.shape[:2]
figure, axes = plt.subplots(1, 3, figsize=(10, 5), layout="constrained")
previews = (frame, gray_images[0].permute(1, 2, 0), images[0].permute(1, 2, 0))
for axis, pixels, title in zip(axes, previews, ("Original", "Gray fill", "Training background")):
    axis.imshow(pixels)
    for cx, cy, bw, bh in boxes:
        axis.add_patch(Rectangle(((cx - bw / 2) * width, (cy - bh / 2) * height),
                                 bw * width, bh * height, fill=False, edgecolor="lime"))
    axis.set_title(title)
    axis.axis("off")
plt.show()

## 2. Mac에서 배경 질감 가림 모델 학습

미리보기를 확인한 뒤 `run_training = True`로 바꾸세요. 같은 사전학습 `yolo26n.pt`에서 시작합니다. 기존 베이스라인과 회색 가림 실행에서 학습 조건을 읽어 대조합니다.

In [ ]:
run_training = False

if run_training:
    data_yaml = prepare_dataset(VIDEOS, ANNOTATIONS / "plate_boxes", SPLIT, DATASET)
    candidate_dir = train_occlusion(data_yaml, baseline_dir, gray_reference=gray_dir)
print("배경 질감 가림 실행 폴더", candidate_dir)

In [ ]:
from ultralytics import YOLO

from wlcv.config import RUNS
from wlcv.training.plate_occlusion import PlateTextureTrainer

last = RUNS / "plate_occlusion_texture" / "weights" / "last.pt"
resume_training = False
if resume_training:
    model = YOLO(str(last))
    model.train(resume=True, trainer=PlateTextureTrainer)

## 3. 원본과 처음 보는 가림 패턴 비교

세 모델 모두 같은 검증 39장을 봅니다. 원본과 원판 박스 중앙 50%를 가린 회색·줄무늬·격자 조건을 비교합니다. 줄무늬와 격자는 평가 코드에서만 생성하며, 세 모델 모두에게 동일한 픽셀로 적용합니다. 실제 가림과 별도로 해석할 수 있도록 a03 2번도 표시합니다.

신뢰도 0.10과 IoU 0.50을 함께 통과한 프레임 수를 비교합니다. 원본의 최고 검증 mAP50–95도 표에 표시합니다. IoU 중앙값은 매칭에 성공한 후보에서 계산합니다.

이 셀은 커널을 다시 시작한 뒤에도 단독으로 실행할 수 있습니다. 다른 실행을 비교할 때는 코드 첫 부분의 `candidate_dir`를 바꾸세요.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from ultralytics import YOLO

from wlcv.config import DATASET, RUNS
from wlcv.evaluation.occlusion_study import validation_occlusion_scores

baseline_dir = RUNS / "plate_baseline"
gray_dir = RUNS / "plate_occlusion"
candidate_dir = RUNS / "plate_occlusion_texture"

runs = {"baseline": baseline_dir, "gray": gray_dir, "texture": candidate_dir}
if all((folder / "weights" / "best.pt").is_file() for folder in runs.values()):
    report_dir = candidate_dir / "report" / "texture_comparison"
    report_dir.mkdir(parents=True, exist_ok=True)
    summaries, frame_scores = [], []
    for name, folder in runs.items():
        model = YOLO(str(folder / "weights" / "best.pt"))
        summary, frames = validation_occlusion_scores(model, DATASET, held_out=True)
        epochs = pd.read_csv(folder / "results.csv", skipinitialspace=True)
        summary["best_val_mAP50-95"] = epochs["metrics/mAP50-95(B)"].max()
        summaries.append(summary.assign(model=name))
        frame_scores.append(frames.assign(model=name))

    comparison = pd.concat(summaries).reset_index()
    details = pd.concat(frame_scores, ignore_index=True)
    comparison.to_csv(report_dir / "summary.csv", index=False)
    details.to_csv(report_dir / "per_frame.csv", index=False)
    display(comparison.round(3))
    display(details.query("video_id == 'a03_sn_f_01' and frame_index == 2").round(3))

    conditions = ["Original", "Plate 50%", "Held-out stripes", "Held-out checker"]
    counts = comparison.pivot(index="condition", columns="model", values="matched_at_threshold")
    axis = counts.reindex(index=conditions, columns=list(runs)).plot.bar(
        rot=0, figsize=(10, 4), color=["#316f9f", "#c77442", "#4b8f79"],
    )
    axis.set_ylabel("Matched frames (conf ≥ 0.10, IoU ≥ 0.50)")
    axis.set_xlabel("")
    axis.set_ylim(0, 43)
    for bars in axis.containers:
        axis.bar_label(bars, padding=2)
    axis.figure.savefig(report_dir / "comparison.png", dpi=180, bbox_inches="tight")
    plt.show()
else:
    print("배경 질감 가림 실험의 학습이 끝난 뒤 이 셀을 실행하세요.")